<a href="https://colab.research.google.com/github/AdrianCurellCruz/World_cup_prediction_using_ML_models/blob/main/EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#EDA

In [1]:
import pandas as pd
import kagglehub
import os


# Download latest version using kaggle API
path = kagglehub.dataset_download("martj42/international-football-results-from-1872-to-2017")
results_file_path = os.path.join(path, "results.csv")
df_all = pd.read_csv(results_file_path)


Using Colab cache for faster access to the 'international-football-results-from-1872-to-2017' dataset.


In [2]:
print(f'Dataset dimensions {df_all.shape}')
print(f'Number of instances: {df_all.shape[0]}')
print(f'Number of variables: {df_all.shape[1]}')
print(df_all.columns.tolist())

Dataset dimensions (49477, 9)
Number of instances: 49477
Number of variables: 9
['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral']


We check if we have NAns in the data

In [3]:
null_counts = df_all.isnull().sum()
print("Null count per column:")
print(null_counts)

Null count per column:
date           0
home_team      0
away_team      0
home_score    44
away_score    44
tournament     0
city           0
country        0
neutral        0
dtype: int64


We then extract all tournaments from the dataset and organize them into distinct hierarchical tiers. This process is essential for the subsequent development of a comprehensive Elo rating system.

In [4]:
tournaments = df_all['tournament'].unique().tolist()
print(tournaments)




['Friendly', 'British Home Championship', 'Évence Coppée Trophy', 'Muratti Vase', 'Copa Lipton', 'Copa Newton', 'Copa Premio Honor Argentino', 'Olympic Games', 'Copa Premio Honor Uruguayo', 'Far Eastern Championship Games', 'Copa Roca', 'Copa América', 'Inter-Allied Games', 'Peace Cup', 'Open International Championship', 'Soccer Ashes', 'Copa Chevallier Boutell', 'Nordic Championship', 'Central European International Cup', 'Baltic Cup', 'Balkan Cup', 'Central American and Caribbean Games', 'FIFA World Cup', 'Copa Rio Branco', 'FIFA World Cup qualification', 'Bolivarian Games', 'CCCF Championship', 'NAFC Championship', 'Copa Oswaldo Cruz', 'Asian Games', 'Pan American Championship', 'Copa del Pacífico', "Copa Bernardo O'Higgins", 'AFC Asian Cup qualification', 'Atlantic Cup', 'AFC Asian Cup', 'African Cup of Nations', 'Copa Paz del Chaco', 'Merdeka Tournament', 'UEFA Euro qualification', 'Southeast Asian Peninsular Games', 'African Friendship Games', 'UEFA Euro', 'Windward Islands Tourn

Subsequently, we categorize the tournaments into distinct tiers based on their relative importance.

#FEATURE ENGINEERING

In [5]:
# Dictionary to store the weight for each tournament
tier_weights = {}

# Iterate through each tournament and assign the weight based on its category
for tournament in tournaments:
    # TIER 1: ONLY the World Cup final stage (Weight 1.0)
    if tournament == 'FIFA World Cup':
        tier_weights[tournament] = 1.0

    # TIER 2: Top continental competitions (Weight 0.9)
    # ONLY the finals of each confederation
    elif tournament in [
        'UEFA Euro',
        'Copa América',
        'African Cup of Nations',
        'AFC Asian Cup',
        'CONCACAF Gold Cup',
        'Oceania Nations Cup'
    ]:
        tier_weights[tournament] = 0.9

    # TIER 3: Intercontinental tournaments, Olympic Games, and elite historical cups (Weight 0.8)
    elif tournament in [
        'Olympic Games',
        'Confederations Cup',
        'CONMEBOL–UEFA Cup of Champions',
        'British Home Championship',
        'Central European International Cup',
        'Balkan Cup',
        'Nordic Championship'
    ]:
        tier_weights[tournament] = 0.8

    # TIER 4: Qualifiers and UEFA Nations League (Weight 0.6)
    elif 'qualification' in tournament.lower():
        tier_weights[tournament] = 0.6

    elif tournament in ['UEFA Nations League']:
        tier_weights[tournament] = 0.6

    # TIER 5: Sub-confederation regional tournaments (Weight 0.4)
    elif tournament in [
        'AFF Championship', 'EAFF Championship', 'SAFF Cup',
        'WAFF Championship', 'COSAFA Cup', 'CECAFA Cup',
        'UNCAF Cup', 'Gulf Cup', 'Arab Cup', 'CAFA Nations Cup',
        'ASEAN Championship', 'Gold Cup qualification',
        'CONCACAF Nations League'
    ]:
        tier_weights[tournament] = 0.4

    # Also detect by prefix (e.g., all tournaments starting with AFF, SAFF, etc.)
    elif tournament.startswith(('AFF', 'EAFF', 'SAFF', 'WAFF', 'COSAFA', 'CECAFA', 'UNCAF')):
        tier_weights[tournament] = 0.4

    # TIER 6: Friendlies, invitationals, minor cups, CONIFA, multi-sport games (Weight 0.2)
    else:
        tier_weights[tournament] = 0.2

# Display the results
print("Tournament weight assignment:")
for tourney, weight in sorted(tier_weights.items(), key=lambda x: x[1], reverse=True):
    print(f"  {tourney}: {weight}")

# Check how many tournaments fall into each weight category
from collections import Counter
weight_counts = Counter(tier_weights.values())
print(f"\nWeight distribution: {dict(weight_counts)}")

Tournament weight assignment:
  FIFA World Cup: 1.0
  Copa América: 0.9
  AFC Asian Cup: 0.9
  African Cup of Nations: 0.9
  UEFA Euro: 0.9
  Oceania Nations Cup: 0.9
  British Home Championship: 0.8
  Olympic Games: 0.8
  Nordic Championship: 0.8
  Central European International Cup: 0.8
  Balkan Cup: 0.8
  CONMEBOL–UEFA Cup of Champions: 0.8
  Confederations Cup: 0.8
  FIFA World Cup qualification: 0.6
  AFC Asian Cup qualification: 0.6
  UEFA Euro qualification: 0.6
  African Cup of Nations qualification: 0.6
  CONCACAF Championship qualification: 0.6
  Copa América qualification: 0.6
  CFU Caribbean Cup qualification: 0.6
  Arab Cup qualification: 0.6
  Oceania Nations Cup qualification: 0.6
  COSAFA Cup qualification: 0.6
  Gold Cup qualification: 0.6
  AFF Championship qualification: 0.6
  AFC Challenge Cup qualification: 0.6
  UEFA Nations League: 0.6
  CONCACAF Nations League qualification: 0.6
  CONIFA World Cup qualification: 0.6
  CONIFA World Football Cup qualification: 0.6

In [6]:
import pandas as pd
import math
from collections import defaultdict

def calculate_elo_ratings(df, tier_weights, base_k=20, home_advantage=60, initial_rating=1500):
    """
    Calculate Elo ratings sequentially for all matches in a DataFrame.

    Parameters:
    - df: DataFrame with columns ['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament']
    - tier_weights: dict mapping tournament names to weights (0.2 to 1.0)
    - base_k: Base K-factor for Elo updates
    - home_advantage: Extra Elo points added to home team
    - initial_rating: Starting Elo for all teams

    Returns:
    - df_with_elo: Original DataFrame with new columns 'elo_home_before' and 'elo_away_before'
    - elo_ratings: Final Elo ratings dictionary (useful for predicting future matches)
    """
    # Sort chronologically (oldest first) to ensure no data leakage
    df_sorted = df.sort_values('date').reset_index(drop=True)

    # Initialize Elo ratings for all teams
    elo_ratings = defaultdict(lambda: initial_rating)

    # Lists to store pre-match ratings
    home_elo_before = []
    away_elo_before = []

    # Iterate through each match chronologically
    for index, row in df_sorted.iterrows():
        home_team = row['home_team']
        away_team = row['away_team']
        home_goals = row['home_score']
        away_goals = row['away_score']
        tournament = row['tournament']

        # Get current Elo ratings (BEFORE the match)
        elo_home = elo_ratings[home_team]
        elo_away = elo_ratings[away_team]

        # Store pre-match ratings for this row
        home_elo_before.append(elo_home)
        away_elo_before.append(elo_away)

        # Adjust home team Elo for home advantage
        elo_home_adj = elo_home + home_advantage

        # Calculate expected win probability (standard Elo formula)
        expected_home = 1 / (1 + 10 ** ((elo_away - elo_home_adj) / 400))
        expected_away = 1 - expected_home

        # Determine actual result (1=win, 0.5=draw, 0=loss)
        if home_goals > away_goals:
            actual_home = 1.0
            actual_away = 0.0
        elif home_goals == away_goals:
            actual_home = 0.5
            actual_away = 0.5
        else:
            actual_home = 0.0
            actual_away = 1.0

        # Margin of victory multiplier (log scale)
        goal_diff = abs(home_goals - away_goals)
        margin_multiplier = math.log(max(goal_diff, 1) + 1)

        # Tournament weight (from tier system)
        weight = tier_weights.get(tournament, 0.2)

        # Calculate K-factor for this match
        K = base_k * weight * margin_multiplier

        # Update Elo ratings (ONLY after the match)
        new_elo_home = elo_home + K * (actual_home - expected_home)
        new_elo_away = elo_away + K * (actual_away - expected_away)

        # Store updated ratings
        elo_ratings[home_team] = new_elo_home
        elo_ratings[away_team] = new_elo_away

    # Add pre-match ratings as new columns
    df_sorted['elo_home_before'] = home_elo_before
    df_sorted['elo_away_before'] = away_elo_before

    return df_sorted, dict(elo_ratings)

Now we are going to implement a new feature H2H wich is the last

In [7]:
import pandas as pd
import numpy as np
from datetime import timedelta

def calculate_weighted_h2h_features(df, half_life_days=1095):  # 3 years ~ 1095 days
    """
    Calculate weighted head-to-head features for each match.
    The weight decays exponentially based on how long ago the match was played.

    Parameters:
    - df: DataFrame with historical matches (must have 'date', 'home_team', 'away_team',
           'home_score', 'away_score')
    - half_life_days: Number of days after which the weight reduces by half (default: 3 years)

    Returns:
    - df with new H2H features added
    """
    # Sort by date to ensure we only look backwards
    df = df.sort_values('date').reset_index(drop=True)

    # Initialize new columns
    h2h_weighted_win_pct = []      # Weighted win percentage for home team against the away team
    h2h_weighted_goal_diff = []    # Weighted average goal difference for home team against away team
    h2h_recency_weight_sum = []    # Sum of weights (to normalize)
    h2h_matches_count = []         # Count of previous meetings (useful for regularization)

    # Convert to datetime if not already
    df['date'] = pd.to_datetime(df['date'])

    # Pre-define decay factor to avoid recalculating e^(-ln2 / half_life) each time
    decay_constant = np.log(2) / half_life_days

    # Iterate through each match (using iterrows for simplicity and clarity)
    for idx, current_match in df.iterrows():
        current_date = current_match['date']
        home_team = current_match['home_team']
        away_team = current_match['away_team']

        # Get ALL previous matches involving these two teams (strictly before current date)
        previous_matches = df[
            (df['date'] < current_date) &
            (
                ((df['home_team'] == home_team) & (df['away_team'] == away_team)) |
                ((df['home_team'] == away_team) & (df['away_team'] == home_team))
            )
        ].copy()

        # If no previous meetings, fill with neutral values (or NaN)
        if len(previous_matches) == 0:
            h2h_weighted_win_pct.append(0.33)   # Neutral 1/3 win probability
            h2h_weighted_goal_diff.append(0.0)
            h2h_recency_weight_sum.append(0.0)
            h2h_matches_count.append(0)
            continue

        # Calculate days difference and weights for each previous match
        days_diff = (current_date - previous_matches['date']).dt.days
        weights = np.exp(-decay_constant * days_diff)  # Exponential decay


        weight_sum = weights.sum()

        # Determine perspective: from Home Team's point of view
        home_team_is_local = (previous_matches['home_team'] == home_team)

        # Goals for the Home team and Away team in those historical matches
        goals_for_home_team = np.where(
            home_team_is_local,
            previous_matches['home_score'],  # If home_team was local, take home_score
            previous_matches['away_score']   # If home_team was away, take away_score
        )
        goals_for_away_team = np.where(
            home_team_is_local,
            previous_matches['away_score'],
            previous_matches['home_score']
        )

        # Goal difference (positive means home_team scored more)
        goal_diff = goals_for_home_team - goals_for_away_team

        # Weighted goal difference
        weighted_goal_diff = np.average(goal_diff, weights=weights)

        # Weighted win percentage for Home Team
        # Result: 1 = win, 0.5 = draw, 0 = loss
        match_results = np.where(
            goals_for_home_team > goals_for_away_team, 1.0,
            np.where(goals_for_home_team == goals_for_away_team, 0.5, 0.0)
        )
        weighted_win_rate = np.average(match_results, weights=weights)

        # Store the results
        h2h_weighted_win_pct.append(weighted_win_rate)
        h2h_weighted_goal_diff.append(weighted_goal_diff)
        h2h_recency_weight_sum.append(weight_sum)
        h2h_matches_count.append(len(previous_matches))

    # Assign to DataFrame
    df['h2h_win_pct'] = h2h_weighted_win_pct
    df['h2h_goal_diff'] = h2h_weighted_goal_diff
    df['h2h_weight_sum'] = h2h_recency_weight_sum
    df['h2h_match_count'] = h2h_matches_count

    return df

In [8]:
def add_elo_difference(df):
    """
    Add the Elo difference feature (home_elo - away_elo) to the DataFrame.

    Parameters:
    - df: DataFrame with columns 'elo_home_before' and 'elo_away_before'

    Returns:
    - df: DataFrame with the new column 'elo_diff'
    """
    # Ensure the required columns exist
    if 'elo_home_before' not in df.columns or 'elo_away_before' not in df.columns:
        raise ValueError("DataFrame must contain 'elo_home_before' and 'elo_away_before' columns")

    df = df.copy()
    df['elo_diff'] = df['elo_home_before'] - df['elo_away_before']

    return df

In [9]:
import pandas as pd
from collections import deque

def add_recent_form(df, window_size=5):
    """
    Add recent form features (Points Per Game over the last N matches) for both home and away teams.
    Calculated strictly using matches played BEFORE the current match date.

    Parameters:
    - df: DataFrame with columns ['date', 'home_team', 'away_team', 'home_score', 'away_score']
    - window_size: Number of previous matches to consider (default: 5)

    Returns:
    - df: DataFrame with new columns 'home_recent_ppg' and 'away_recent_ppg'
    """
    # Sort chronologically to ensure backward-looking calculation
    df_sorted = df.sort_values('date').reset_index(drop=True)

    # Dictionary to store recent history for each team
    # Each value is a deque of tuples: (points_earned, goal_diff)
    recent_history = defaultdict(lambda: deque(maxlen=window_size))

    home_recent_ppg = []
    away_recent_ppg = []

    for index, row in df_sorted.iterrows():
        home_team = row['home_team']
        away_team = row['away_team']
        home_goals = row['home_score']
        away_goals = row['away_score']

        # Get recent history for both teams (ONLY past matches, nothing current)
        home_history = recent_history[home_team]
        away_history = recent_history[away_team]

        # Calculate Points Per Game (PPG) for home team
        if len(home_history) == 0:
            home_ppg = 1.5  # Neutral baseline (mid-point between 0 and 3)
        else:
            total_points = sum(points for points, _ in home_history)
            home_ppg = total_points / len(home_history)

        # Calculate PPG for away team
        if len(away_history) == 0:
            away_ppg = 1.5
        else:
            total_points = sum(points for points, _ in away_history)
            away_ppg = total_points / len(away_history)

        home_recent_ppg.append(home_ppg)
        away_recent_ppg.append(away_ppg)

        # Determine points earned in THIS match (from each team's perspective)
        if home_goals > away_goals:
            home_points = 3
            away_points = 0
        elif home_goals == away_goals:
            home_points = 1
            away_points = 1
        else:
            home_points = 0
            away_points = 3

        # Goal difference for this match (from each team's perspective)
        home_goal_diff = home_goals - away_goals
        away_goal_diff = away_goals - home_goals

        # Update history with this match's data (so it's available for FUTURE matches)
        recent_history[home_team].append((home_points, home_goal_diff))
        recent_history[away_team].append((away_points, away_goal_diff))

    # Add the new columns to the DataFrame
    df_sorted['home_recent_ppg'] = home_recent_ppg
    df_sorted['away_recent_ppg'] = away_recent_ppg

    return df_sorted

In [10]:
def add_recent_goal_average(df, window_size=5):
    """
    Add recent goal average features (average goal difference over the last N matches)
    for both home and away teams. Calculated strictly using matches played BEFORE the current match.

    Parameters:
    - df: DataFrame with columns ['date', 'home_team', 'away_team', 'home_score', 'away_score']
    - window_size: Number of previous matches to consider (default: 5)

    Returns:
    - df: DataFrame with new columns 'home_recent_goal_avg' and 'away_recent_goal_avg'
    """
    # Sort chronologically to ensure backward-looking calculation
    df_sorted = df.sort_values('date').reset_index(drop=True)

    # Dictionary to store recent goal differences for each team
    recent_goal_diffs = defaultdict(lambda: deque(maxlen=window_size))

    home_recent_goal_avg = []
    away_recent_goal_avg = []

    for index, row in df_sorted.iterrows():
        home_team = row['home_team']
        away_team = row['away_team']
        home_goals = row['home_score']
        away_goals = row['away_score']

        # Get recent goal differences for both teams (ONLY past matches)
        home_diffs = recent_goal_diffs[home_team]
        away_diffs = recent_goal_diffs[away_team]

        # Calculate average goal difference for home team
        if len(home_diffs) == 0:
            home_goal_avg = 0.0  # Neutral baseline
        else:
            home_goal_avg = sum(home_diffs) / len(home_diffs)

        # Calculate average goal difference for away team
        if len(away_diffs) == 0:
            away_goal_avg = 0.0
        else:
            away_goal_avg = sum(away_diffs) / len(away_diffs)

        home_recent_goal_avg.append(home_goal_avg)
        away_recent_goal_avg.append(away_goal_avg)

        # Goal difference from each team's perspective
        home_goal_diff = home_goals - away_goals
        away_goal_diff = away_goals - home_goals

        # Update history with this match's data (for future matches)
        recent_goal_diffs[home_team].append(home_goal_diff)
        recent_goal_diffs[away_team].append(away_goal_diff)

    # Add the new columns to the DataFrame
    df_sorted['home_recent_goal_avg'] = home_recent_goal_avg
    df_sorted['away_recent_goal_avg'] = away_recent_goal_avg

    return df_sorted

In [11]:
def add_target_variable(df):
    """
    Add the target variable (match result from the home team's perspective).

    Encoding:
    - 0: Draw
    - 1: Home Win
    - 2: Away Win

    Parameters:
    - df: DataFrame with columns 'home_score' and 'away_score'

    Returns:
    - df: DataFrame with new column 'target'
    """
    df = df.copy()

    def get_result(row):
        if row['home_score'] > row['away_score']:
            return 1  # Home Win
        elif row['home_score'] == row['away_score']:
            return 0  # Draw
        else:
            return 2  # Away Win

    df['target'] = df.apply(get_result, axis=1)

    return df

In [12]:
import pandas as pd
import math
from collections import defaultdict


# Calculate Elo ratings (uses only past data)
df_full_with_elo, final_elo_ratings = calculate_elo_ratings(
    df=df_all,
    tier_weights=tier_weights,
    base_k=20,
    home_advantage=60
)

# Add Elo Difference feature
df_full_features = add_elo_difference(df_full_with_elo)

# Add Recent Form (Points Per Game over last 5 matches)
df_full_features = add_recent_form(df_full_features, window_size=5)

# Add Recent Goal Average over last 5 matches
df_full_features = add_recent_goal_average(df_full_features, window_size=5)

# Add the Target variable (0=Draw, 1=Home Win, 2=Away Win)
df_full_features = add_target_variable(df_full_features)

# Split chronologically (NO SHUFFLE!) to avoid data leakage
cutoff_date = '2022-11-20'
df_train = df_full_features[df_full_features['date'] < cutoff_date]
df_test = df_full_features[df_full_features['date'] >= cutoff_date]



# Prepare features and target for the model
feature_cols = [
    'elo_diff',
    'elo_home_before',
    'elo_away_before',
    'home_recent_ppg',
    'away_recent_ppg',
    'home_recent_goal_avg',
    'away_recent_goal_avg'
]

X_train = df_train[feature_cols].fillna(0)
y_train = df_train['target']

X_test = df_test[feature_cols].fillna(0)
y_test = df_test['target']

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")



Check: Argentina's form before the 2022 World Cup (should use matches BEFORE Nov 2022):
             date  home_recent_ppg  home_recent_goal_avg
45721  2022-11-22              3.0                   3.8
45737  2022-11-26              2.4                   3.0
45765  2022-12-03              2.4                   2.2
Train shape: (45700, 7), Test shape: (3777, 7)


#MODELING

##RANDOM FOREST

###Baseline

In [25]:
# Train and evaluate the model
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

model = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"\nAccuracy on 2022 World Cup: {accuracy * 100:.2f}%")
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix (rows=actual, cols=predicted):")
print(cm)
print("   (Order: Draw, Home Win, Away Win)")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Draw', 'Home Win', 'Away Win']))


Accuracy on 2022 World Cup: 58.43%

Confusion Matrix (rows=actual, cols=predicted):
[[   2  584  275]
 [   7 1570  188]
 [   6  510  635]]
   (Order: Draw, Home Win, Away Win)

Classification Report:
              precision    recall  f1-score   support

        Draw       0.13      0.00      0.00       861
    Home Win       0.59      0.89      0.71      1765
    Away Win       0.58      0.55      0.56      1151

    accuracy                           0.58      3777
   macro avg       0.43      0.48      0.43      3777
weighted avg       0.48      0.58      0.50      3777



##XGB BOOSTING

###Baseline

In [28]:
from xgboost import XGBClassifier

# Baseline XGBoost
xgb_baseline = XGBClassifier(random_state=42)
xgb_baseline.fit(X_train, y_train)
y_pred_xgb = xgb_baseline.predict(X_test)
accuracy_baseline=accuracy_score(y_test, y_pred_xgb)
print(f"XGBoost Baseline Accuracy: {accuracy_score(y_test, y_pred_xgb) * 100:.2f}%")

XGBoost Baseline Accuracy: 57.40%


##CAT BOOSTING


In [17]:
!pip3 install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 6.0 MB/s eta 0:00:00


###Baseline

In [18]:
from catboost import CatBoostClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import numpy as np
import pandas as pd
from scipy.stats import randint, uniform



# Instanciate baseline model with default settings
baseline_model = CatBoostClassifier(verbose=0, random_seed=42)

# Train on historical data (pre-2022)
baseline_model.fit(X_train, y_train)

# Predict on 2022 World Cup
y_pred_baseline = baseline_model.predict(X_test)
y_pred_proba_baseline = baseline_model.predict_proba(X_test)

# Metrics
accuracy_baseline = accuracy_score(y_test, y_pred_baseline)

print(f"Baseline Accuracy (2022 World Cup): {accuracy_baseline * 100:.2f}%")
print("\nConfusion Matrix (Baseline):")
print(confusion_matrix(y_test, y_pred_baseline))
print("\nClassification Report (Baseline):")
print(classification_report(y_test, y_pred_baseline, target_names=['Draw', 'Home Win', 'Away Win']))

STEP 1: TRAINING CATBOOST BASELINE (Default parameters)
--------------------------------------------------
Baseline Accuracy (2022 World Cup): 57.72%

Confusion Matrix (Baseline):
[[  27  577  257]
 [  34 1545  186]
 [  57  486  608]]

Classification Report (Baseline):
              precision    recall  f1-score   support

        Draw       0.23      0.03      0.06       861
    Home Win       0.59      0.88      0.71      1765
    Away Win       0.58      0.53      0.55      1151

    accuracy                           0.58      3777
   macro avg       0.47      0.48      0.44      3777
weighted avg       0.51      0.58      0.51      3777



##HISTORY GRADIENT BOOSTING

###Baseline

In [24]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# HISTGRADIENTBOOSTING BASELINE (Default parameters)
print("TRAINING HISTGRADIENTBOOSTING BASELINE (Default parameters)")

# Instantiate the model with default parameters
# random_state=42 ensures reproducibility
# default parameters: max_iter=100, max_depth=None, learning_rate=0.1
hgb_model = HistGradientBoostingClassifier(random_state=42)

# Train on historical data (pre-2022)
hgb_model.fit(X_train, y_train)

# Predict on 2022 World Cup
y_pred_hgb = hgb_model.predict(X_test)
y_pred_proba_hgb = hgb_model.predict_proba(X_test)

# Calculate metrics
accuracy_hgb = accuracy_score(y_test, y_pred_hgb)

print(f"Baseline Accuracy (2022 World Cup): {accuracy_hgb * 100:.2f}%")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_hgb))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_hgb, target_names=['Draw', 'Home Win', 'Away Win']))




TRAINING HISTGRADIENTBOOSTING BASELINE (Default parameters)
--------------------------------------------------
Baseline Accuracy (2022 World Cup): 58.96%

Confusion Matrix:
[[   6  588  267]
 [   1 1578  186]
 [   7  501  643]]

Classification Report:
              precision    recall  f1-score   support

        Draw       0.43      0.01      0.01       861
    Home Win       0.59      0.89      0.71      1765
    Away Win       0.59      0.56      0.57      1151

    accuracy                           0.59      3777
   macro avg       0.54      0.49      0.43      3777
weighted avg       0.55      0.59      0.51      3777

